<img src="../assets/ga-logo.png" style="float: left; margin: 20px; height: 55px">

# Using `matplotlib` and `seaborn` - Solution

---

### About
Understand `matplotlib`'s object-oriented interface and create heatmaps with `seaborn`.

### Learning Objectives
- Understand the object-oriented way of working with `matplotlib` to fully control your visualizations
- Generate heat maps in `seaborn`

### Notebook Guide
- Scenario
- Different ways to use `matplotlib`
- Introducing `seaborn`
- Now You Try!
- Conclusions and Takeaways

# Scenario

You're an analyst working for CoolBricks - a new toy company who wants to compete with LEGO by producing their own bricks and sets. To better understand what kind of sets to create and at what price points, you are tasked with analyzing LEGO's catalogue over time.

The data comes from Kaggle: [https://www.kaggle.com/datasets/alexracape/lego-sets-and-prices-over-time](https://www.kaggle.com/datasets/alexracape/lego-sets-and-prices-over-time)

We are going to use `pandas` to read this data and `matplotlib` and `seaborn` to create plots.

Our research question is: ***How do product characteristics (price, number of pieces) vary across different themes and categories?***

# Different ways to use `matplotlib`
---

Part of `matplotlib`'s origin story is replicating how MATLAB creates plots. This is considered the old way of using `matplotlib` but you'll still see that code in places.

That code looks something like this:

```python
import matplotlib.pyplot as plt

plt.scatter(x=df["column_1"], y=df["column_2"])

plt.title("This is a scatter plot")
```

This is fine, but you don't have full control of your plots. The best you can do is retrieve the "most recent" plot that was created under the hood.

A better way is to use the object-oriented interface.

This means:

- Creating a blank figure (the entire chart area) and one or more axis (an individual plot)
- Using these figure and axis objects to draw your plot(s)
- Controlling every aspect of these figures and axes (titles, colors, fonts, gridlines etc.)

*Note: in `matplotlib`, every chart consists of one `Figure` and one or more `Axis` objects*

In [ ]:
import pandas as pd

Let's look at the data.

In [ ]:
lego = pd.read_csv("../data/lego.csv")
lego.head()

Let's look at the breakdown of `Category`.

In [ ]:
lego["Category"].value_counts()

To create a bar chart, we can use the `pandas` `.plot()` function (which uses `matplotlib` under the hood)

In [ ]:
lego["Category"].value_counts().sort_values().plot(kind="barh");

We can change some options directly in `.plot()` but to have full control, we will:

- Reference `matplotlib` directly
- Create a `Figure` and `Axis` object
- Tell the `.plot()` method to use our own `Figure` and `Axis` objects rather than create its own

In [ ]:
import matplotlib.pyplot as plt

fig, axis = plt.subplots()

print(type(fig))
print(type(axis))

In [ ]:
fig, axis = plt.subplots()

(
    lego["Category"]
    .value_counts()
    .sort_values()  # to ensure bars are in the right order
    .plot(kind="barh", ax=axis)  # notice the additional parameter `ax`
)

# now we set all options on the Axis object
axis.set_title("Most products are categorized as 'Normal'")

# or we can use the generic `.set()` method`
axis.set(xlabel="Number of products", ylabel="Product category");

You can then even save your figure object to file!

In [ ]:
fig.savefig("my_chart.png")

# Introducing `seaborn`
---

Sometimes, we hit the limit of what `matplotlib` can do when we want more advanced visualizations.

One library that works much like `matplotlib` but has more advanced chart types is [`seaborn`](https://seaborn.pydata.org/).

A typical use case is creating scatter plots colored by a third variable. This is possible in `matplotlib` but much easier in `seaborn`.

In [ ]:
import seaborn as sns

sns.scatterplot(
    lego,
    x="Pieces",
    y="USD_MSRP",
    hue="Theme_Group",  # a third variable to color each point
    alpha=0.6,
);

Seaborn also does nicer versions of box plots, histograms etc.

In [ ]:
fig, axis = plt.subplots()

sns.boxplot(lego, x="USD_MSRP", y="Theme_Group", ax=axis)

axis.set(title="Distribution of price by theme group", ylabel=None);

In [ ]:
fig, axis = plt.subplots(figsize=(14, 6))

sns.histplot(lego[lego["USD_MSRP"] < 100], x="USD_MSRP", hue="Category", ax=axis)

axis.set(
    title="Distribution of price by product category (under $100)", ylabel="Frequency"
);

Another use case is a heatmap. Let's look at the number of products per year per category (beyond 2010 so we don't have too many columns).

In [ ]:
lego_recent = lego[lego["Year"] >= 2010]
pd.crosstab(lego_recent["Category"], lego_recent["Year"])

We can just about work out what's happening here, but `seaborn`'s heatmap lets us see patterns much more easily.

In [ ]:
sns.heatmap(
    data=pd.crosstab(lego_recent["Category"], lego_recent["Year"]),
);

We might want to change some of the default options:

- the colormap (the range of colors used in the scale) should be perhaps a single color from light to dark
- we can use `matplotlib` objects to control the axes, add titles etc.

*Note: read more about colormaps here: [`matplotlib` colormaps](https://matplotlib.org/stable/users/explain/colors/colormaps.html)*

In [ ]:
fig, axis = plt.subplots()

sns.heatmap(
    data=pd.crosstab(lego_recent["Category"], lego_recent["Year"]),
    cmap="Blues",
    ax=axis,
)

axis.set(
    title="The Gear category was popular but these days most products are the 'Normal' category",
    ylabel="Product category",
);

# Now You Try!
---
1. Create a crosstab to compare how many sets are in each category and theme group combinations. Have categories at the top and one row per theme group.

In [ ]:
category_theme_table = pd.crosstab(
    index=lego["Theme_Group"],
    columns=lego["Category"],
)

category_theme_table

2. Create a `seaborn` heatmap to visualize this information. Remember to change:

- the default color map
- the axis labels and chart title (using `matplotlib` objects)

In [ ]:
fig, axis = plt.subplots()

sns.heatmap(
    data=category_theme_table,
    cmap="Greens",
)

axis.set(
    title="Most products are categorized as 'Normal',\nbut some categories are spread across categories more than others",
    ylabel="Theme group",
    xlabel="Product category",
);

3. *BONUS:* Work out how to change the `crosstab` to *normalize* each row, so the values represent the % of records that each product is categorized as within a theme. That is: within each theme group, what percentage of products fall in each category? Each row should add up to 100%.

Visualize this information as a heatmap.

In [ ]:
fig, axis = plt.subplots()

category_theme_table_normalized = (
    pd.crosstab(
        index=lego["Theme_Group"],
        columns=lego["Category"],
        normalize="index",
    )
    * 100
)

sns.heatmap(
    data=category_theme_table_normalized,
    cmap="Greens",
    vmin=0,
    vmax=100,  # numbers should go from 0-100%
    annot=True,  # add annotations
    fmt=".1f",  # show values to 1 decimal place
)

axis.set(
    title="Most theme groups are nearly 100% 'Normal' products.\nA notable exception is 'Contstraction'",
    ylabel="Theme group",
    xlabel="Product category",
);

# Conclusions and Takeaways

- The object-oriented interface is the preferred way to use `matplotlib`
- Creating `matplotlib` objects directly gives us full control over our plots
- For more advanced visualizations, `seaborn` offers a greater variety of chart types